In [1]:
import json, os
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession.builder
    .appName("reference")
    .master("local[*]")
    .config("spark.driver.memory", "5g")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate()
)

In [3]:
S = "file:///data-lake/data-lake/silver"
keys = {
    "customers": [r[0] for r in spark.read.parquet(S + "/DimCustomer").select("CustomerKey").collect()],
    "stores":    [r[0] for r in spark.read.parquet(S + "/DimStore").select("StoreKey").collect()],
    "products":  {str(r[0]): float(r[1]) for r in
                  spark.read.parquet(S + "/DimProduct").select("ProductKey", "UnitPrice").collect() if r[1]},
}
os.makedirs("/data-lake/data-lake/reference", exist_ok=True)
json.dump(keys, open("/data-lake/data-lake/reference/event_keys.json", "w"))

In [4]:
spark.stop()